In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv(
    "exchange_bias_baseline.csv"
)

# ==========================================
# TARGET
# ==========================================

TARGET = "exc_bias_oe"

X = df.drop(
    columns=[TARGET]
)

y = df[TARGET]

# ==========================================
# TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# ==========================================
# RANDOM FOREST
# ==========================================

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train,
    y_train
)

# ==========================================
# PREDICTION
# ==========================================

y_pred = rf.predict(
    X_test
)

# ==========================================
# METRICS
# ==========================================

r2 = r2_score(
    y_test,
    y_pred
)

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = mean_squared_error(
    y_test,
    y_pred
) ** 0.5

print("\nResults")
print("R2   :", round(r2,4))
print("MAE  :", round(mae,4))
print("RMSE :", round(rmse,4))


Results
R2   : 0.5061
MAE  : 190.8522
RMSE : 296.6976


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from xgboost import XGBRegressor

# =====================================
# LOAD DATA
# =====================================

df = pd.read_csv(
    "exchange_bias_enhanced.csv"
)

TARGET = "exc_bias_oe"

X = df.drop(columns=[TARGET])
y = df[TARGET]

# =====================================
# SPLIT
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# =====================================
# XGBOOST
# =====================================

model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

# =====================================
# PREDICTION
# =====================================

y_pred = model.predict(X_test)

# =====================================
# METRICS
# =====================================

r2 = r2_score(y_test, y_pred)

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

print("\nResults")
print("R2   :", round(r2,4))
print("MAE  :", round(mae,4))
print("RMSE :", round(rmse,4))


Results
R2   : 0.4997
MAE  : 191.337
RMSE : 298.6156


In [3]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print(
    importance.head(25)
)

                                       feature  importance
17                                     shell_b    0.078588
31   core_pauling_electronegativity_difference    0.064854
28                      core_sum_work_function    0.052103
27                                core_sum_vec    0.051761
25              core_pauling_electronegativity    0.042244
19                                  shell_beta    0.038544
5                                      coer_oe    0.036780
36              shell_work_function_difference    0.036313
33               core_work_function_difference    0.036278
35                        shell_vec_difference    0.036259
3                                temperature_k    0.032524
20                                 shell_gamma    0.030468
34  shell_pauling_electronegativity_difference    0.026711
10                                  fc_field_t    0.026159
16                                  core_gamma    0.024585
21                            num_of_magn_ions    0.0238

In [5]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# ====================================
# LOAD PAPER DATASETS
# ====================================

train_df = pd.read_csv(
    "data/final_data_exchange_bias.csv"
)

valid_df = pd.read_csv(
    "data/validation_dataset.csv"
)

TARGET = "exc_bias_oe"

# ====================================
# REMOVE NON-NUMERIC COLUMNS
# ====================================

drop_cols = [
    "core",
    "shell",
    "formula"
]

X_train = train_df.drop(
    columns=drop_cols + [TARGET]
)

y_train = train_df[TARGET]

X_valid = valid_df.drop(
    columns=drop_cols + [TARGET, "link doi"],
    errors="ignore"
)

y_valid = valid_df[TARGET]

# ====================================
# MODEL
# ====================================

model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

# ====================================
# VALIDATION
# ====================================

pred = model.predict(X_valid)

r2 = r2_score(
    y_valid,
    pred
)

mae = mean_absolute_error(
    y_valid,
    pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        pred
    )
)

print("R2   :", r2)
print("MAE  :", mae)
print("RMSE :", rmse)

R2   : 0.9967801165805406
MAE  : 14.463471727743336
RMSE : 18.952189609563995


In [6]:
train_pairs = set(
    train_df["formula"].astype(str)
)

valid_pairs = set(
    valid_df["formula"].astype(str)
)

overlap = train_pairs.intersection(
    valid_pairs
)

print("Overlap:", len(overlap))

print(list(overlap)[:50])

Overlap: 39
['NiCoO-NiCo', 'NiO-NiFe2O4-Fe2O3', 'Fe2O3', 'TiO2-Fe2O3', 'FeRh', 'BiFe0.5Sc0.5O3', 'Fe-Fe3O4', 'Mn3O4', 'Fe-MnO2', 'SmMnO3-Mn2O3', 'CoFe2O4-BiFeO3', 'Co-CoO', 'ZnO', 'Co-Ni', 'CoFe2O4-Fe3O4', 'Ga-Fe2O3', 'MnFe2O4-Fe3O4', 'CoO-CoFe', 'NiO-Ni0.05Mn0.95O', 'Mn2O3', 'Co0.96Fe0.04-Co0.99Fe1.01O4', 'Fe3O4-CoFe2O4', 'CoO-Fe3O4', 'Co-Co3O4', 'MnFe2O4-Fe2O3', 'CuCr2O4-Cr2O3', 'BiFeO3', 'CoO-Au', 'Ag-Fe3O4', 'Sm0.5Ca0.5MnO3-CO', 'Fe3O4-Fe2O3', 'NiO-Mn3O4', 'CoFe2O4-NiO', 'FeO-Fe-ZnO', 'Cr2O3-CoFe2O4', 'La0.67Sr0.33MnO3', 'Ni3Al-NiO', 'CoCr2O4', 'NiO-NiO']


In [7]:
validation_formulas = set(
    valid_df["formula"].astype(str)
)

train_filtered = train_df[
    ~train_df["formula"].astype(str).isin(
        validation_formulas
    )
]

In [8]:
print(train_df.shape)
print(train_filtered.shape)

(979, 41)
(549, 41)


In [9]:
remaining_overlap = set(
    train_filtered["formula"].astype(str)
).intersection(
    set(valid_df["formula"].astype(str))
)

print(len(remaining_overlap))

0


In [12]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# =========================================
# TARGET
# =========================================

TARGET = "exc_bias_oe"

drop_cols = [
    "core",
    "shell",
    "formula"
]

# =========================================
# TRAIN DATA
# =========================================

X_train = train_filtered.drop(
    columns=drop_cols + [TARGET],
    errors="ignore"
)

y_train = train_filtered[TARGET]

# =========================================
# VALIDATION DATA
# =========================================

X_valid = valid_df.drop(
    columns=drop_cols + [TARGET, "link doi"],
    errors="ignore"
)

y_valid = valid_df[TARGET]

print("Train shape :", X_train.shape)
print("Valid shape :", X_valid.shape)

# =========================================
# XGBOOST
# =========================================

model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# =========================================
# TRAIN
# =========================================

model.fit(
    X_train,
    y_train
)

# =========================================
# PREDICT
# =========================================

pred = model.predict(
    X_valid
)

# =========================================
# METRICS
# =========================================

r2 = r2_score(
    y_valid,
    pred
)

mae = mean_absolute_error(
    y_valid,
    pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        pred
    )
)

print("\nRESULTS")
print("R2   :", round(r2,4))
print("MAE  :", round(mae,4))
print("RMSE :", round(rmse,4))

# =========================================
# FEATURE IMPORTANCE
# =========================================

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print("\nTOP 25 FEATURES")
print(
    importance.head(25)
)

Train shape : (549, 37)
Valid shape : (51, 37)

RESULTS
R2   : 0.0697
MAE  : 257.6335
RMSE : 322.152

TOP 25 FEATURES
                                       feature  importance
31   core_pauling_electronegativity_difference    0.084518
25              core_pauling_electronegativity    0.054624
36              shell_work_function_difference    0.046974
5                                      coer_oe    0.042829
17                                     shell_b    0.042058
35                        shell_vec_difference    0.039465
33               core_work_function_difference    0.038771
30                     shell_sum_work_function    0.037235
34  shell_pauling_electronegativity_difference    0.034075
15                                   core_beta    0.033890
28                      core_sum_work_function    0.032215
13                                      core_b    0.031690
14                                  core_alpha    0.031016
1                              h_range_max_koe    0.0302

In [13]:
overlap_counts = (
    train_df[
        train_df["formula"].isin(validation_formulas)
    ]
    ["formula"]
    .value_counts()
)

print(overlap_counts.head(50))

formula
Co-CoO                         61
NiO-NiO                        47
Fe-Fe3O4                       22
Fe2O3                          20
CoO-Fe3O4                      20
SmMnO3-Mn2O3                   20
Co-Co3O4                       19
BiFeO3                         16
Fe3O4-Fe2O3                    15
CoFe2O4-NiO                    14
Sm0.5Ca0.5MnO3-CO              14
TiO2-Fe2O3                     13
CoFe2O4-Fe3O4                  13
CoCr2O4                        12
Fe3O4-CoFe2O4                  10
NiCoO-NiCo                      9
Ga-Fe2O3                        8
CoFe2O4-BiFeO3                  8
MnFe2O4-Fe2O3                   8
Cr2O3-CoFe2O4                   8
FeRh                            8
La0.67Sr0.33MnO3                7
Co0.96Fe0.04-Co0.99Fe1.01O4     7
Mn2O3                           7
NiO-Mn3O4                       6
BiFe0.5Sc0.5O3                  6
FeO-Fe-ZnO                      4
NiO-NiFe2O4-Fe2O3               4
Mn3O4                           3
CoO-Co